In [2]:
!pip install -q matplotlib seaborn scikit-learn

In [3]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!cp -r '/content/drive/MyDrive/Colab Notebooks/food_dataset_complete' /content/food_dataset_complete

In [5]:
import os

DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/food_dataset_complete"

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VALIDATE_DIR = os.path.join(DATASET_DIR, 'val')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

print("=======Dataset Summary=======")

for split in ['train', 'val', 'test']:
    print("\n", split)

    path = os.path.join(DATASET_DIR, split)

    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)


        if not os.path.isdir(cls_path):
            continue

        count = len(os.listdir(cls_path))
        print(cls, ":", count, "photo")

=======Dataset Summary=======

 train
thaisourcurry : 103 photo
tomyum : 104 photo
padthai : 167 photo
somtam : 188 photo
panangchickencurry : 105 photo
larb : 102 photo
tomkhagai : 131 photo
padkrapao : 180 photo
massamancurry : 103 photo
greencurry : 176 photo

 val
panangchickencurry : 49 photo
larb : 47 photo
padkrapao : 62 photo
tomkhagai : 59 photo
tomyum : 49 photo
padthai : 61 photo
thaisourcurry : 55 photo
massamancurry : 46 photo
greencurry : 69 photo
somtam : 68 photo

 test
massamancurry : 19 photo
greencurry : 32 photo
padthai : 28 photo
larb : 20 photo
thaisourcurry : 19 photo
panangchickencurry : 20 photo
padkrapao : 32 photo
tomyum : 20 photo
tomkhagai : 23 photo
somtam : 32 photo


In [6]:
IMG_SIZE = 224           #  ใช้ 224x224
BATCH_SIZE  = 32         # แบ่งรูปเทรนต่อรอบ
EPOCHS_TOP  = 30
EPOCHS_FINE = 20         # Fine-tune ทั้ง model
LR_TOP      = 5e-4       # Learning rate ช่วงแรก
LR_FINE     = 1e-4       # Learning rate ช่วง fine-tune
MODEL_PATH = '/content/food_classifier.h5'

CLASS_NAME = ['greencurry', 'larb', 'massamancurry', 'padkrapao',
              'padthai', 'panangchickencurry', 'somtam',
              'thaisourcurry', 'tomkhagai', 'tomyum']

NUM_CLASS = len(CLASS_NAME)

print("=======Config=======")
print(f' Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f' Batch size : {BATCH_SIZE}')
print(f' Epochs top : {EPOCHS_TOP}')
print(f' Epochs fine: {EPOCHS_FINE}')
print(f' Learning rate top : {LR_TOP}')
print(f' Learning rate fine: {LR_FINE}')
print(f' Amout Class  : {NUM_CLASS}')
print(f' Class name : {CLASS_NAME}')

=======Config=======
 Image size  : 224x224
 Batch size : 32
 Epochs top : 30
 Epochs fine: 20
 Learning rate top : 0.0005
 Learning rate fine: 0.0001
 Amout Class  : 10
 Class name : ['greencurry', 'larb', 'massamancurry', 'padkrapao', 'padthai', 'panangchickencurry', 'somtam', 'thaisourcurry', 'tomkhagai', 'tomyum']


In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import preprocess_input
import os

TEST_DIR = os.path.join(DATASET_DIR, 'test')

train_data = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = 'nearest'
)

val_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

test_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

# load
train_loader = train_data.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle=True
)

val_loader = val_data.flow_from_directory(
    VALIDATE_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

test_loader = test_data.flow_from_directory(
    TEST_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

print('Class indices:', train_loader.class_indices)

Found 1358 images belonging to 10 classes.
Found 565 images belonging to 10 classes.
Found 245 images belonging to 10 classes.
Class indices: {'greencurry': 0, 'larb': 1, 'massamancurry': 2, 'padkrapao': 3, 'padthai': 4, 'panangchickencurry': 5, 'somtam': 6, 'thaisourcurry': 7, 'tomkhagai': 8, 'tomyum': 9}


In [8]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import layers, models

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

features = base_model.output
x = layers.GlobalAveragePooling2D()(features)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

predictions = layers.Dense(NUM_CLASS, activation='softmax')(x)
food_model = models.Model(inputs=base_model.input, outputs=predictions)

food_model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,302,474 (27.86 MB)

 Trainable params: 264,970 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

# **TRAIN**

In [9]:
import tensorflow as tf

food_model.compile (
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_TOP),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

save_best = tf.keras.callbacks.ModelCheckpoint (
    MODEL_PATH,
    monitor='val_accuracy',
    save_best_only=True,
)

stop_early = tf.keras.callbacks.EarlyStopping (
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    verbose=1
)

print("=======Start Training=======")

train_result = food_model.fit (
    train_loader,
    validation_data = val_loader,
    epochs = EPOCHS_TOP,
    callbacks = [save_best, stop_early, reduce_lr]
)


=======Start Training=======
Epoch 1/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2353 - loss: 2.2130

43/43 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.3255 - loss: 1.9304 - val_accuracy: 0.6708 - val_loss: 1.2572 - learning_rate: 5.0000e-04
Epoch 2/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 750ms/step - accuracy: 0.5673 - loss: 1.2943

43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 910ms/step - accuracy: 0.6060 - loss: 1.2031 - val_accuracy: 0.7327 - val_loss: 0.9199 - learning_rate: 5.0000e-04
Epoch 3/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 760ms/step - accuracy: 0.7100 - loss: 0.9324

43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 906ms/step - accuracy: 0.7275 - loss: 0.8794 - val_accuracy: 0.8000 - val_loss: 0.6498 - learning_rate: 5.0000e-04
Epoch 4/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 790ms/step - accuracy: 0.7511 - loss: 0.7839

43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 935ms/step - accuracy: 0.7629 - loss: 0.7464 - val_accuracy: 0.8071 - val_loss: 0.6193 - learning_rate: 5.0000e-04
Epoch 5/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 755ms/step - accuracy: 0.7823 - loss: 0.6807

...

43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 945ms/step - accuracy: 0.8733 - loss: 0.3826 - val_accuracy: 0.9274 - val_loss: 0.2554 - learning_rate: 5.0000e-04
Epoch 15/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 924ms/step - accuracy: 0.8954 - loss: 0.3269 - val_accuracy: 0.9009 - val_loss: 0.2876 - learning_rate: 5.0000e-04
Epoch 16/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 759ms/step - accuracy: 0.8888 - loss: 0.3285
Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
43/43 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.8837 - loss: 0.3371 - val_accuracy: 0.9168 - val_loss: 0.2558 - learning_rate: 5.0000e-04
Epoch 17/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 908ms/step - accuracy: 0.9146 - loss: 0.2882 - val_accuracy: 0.9239 - val_loss: 0.2374 - learning_rate: 1.5000e-04
Epoch 18/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 898ms/step - accuracy: 0.9153 - loss: 0.2732 - val_accuracy: 0.9186 - val_loss: 0.2363 - learning_rate: 1.5000e-04
Epoch 19/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.9225 -

43/43 ━━━━━━━━━━━━━━━━━━━━ 41s 949ms/step - accuracy: 0.9234 - loss: 0.2718 - val_accuracy: 0.9327 - val_loss: 0.2093 - learning_rate: 1.5000e-04
Epoch 20/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 930ms/step - accuracy: 0.9153 - loss: 0.2767 - val_accuracy: 0.9310 - val_loss: 0.2166 - learning_rate: 1.5000e-04
Epoch 21/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 757ms/step - accuracy: 0.9330 - loss: 0.2290
Epoch 21: ReduceLROnPlateau reducing learning rate to 4.500000213738531e-05.
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 909ms/step - accuracy: 0.9183 - loss: 0.2576 - val_accuracy: 0.9257 - val_loss: 0.2198 - learning_rate: 1.5000e-04
Epoch 22/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 772ms/step - accuracy: 0.9300 - loss: 0.2306

43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 914ms/step - accuracy: 0.9271 - loss: 0.2334 - val_accuracy: 0.9345 - val_loss: 0.2107 - learning_rate: 4.5000e-05
Epoch 23/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 915ms/step - accuracy: 0.9205 - loss: 0.2476 - val_accuracy: 0.9327 - val_loss: 0.2074 - learning_rate: 4.5000e-05
Epoch 24/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 772ms/step - accuracy: 0.9213 - loss: 0.2714

43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 934ms/step - accuracy: 0.9212 - loss: 0.2590 - val_accuracy: 0.9363 - val_loss: 0.2042 - learning_rate: 4.5000e-05
Epoch 25/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 919ms/step - accuracy: 0.9190 - loss: 0.2583 - val_accuracy: 0.9363 - val_loss: 0.2037 - learning_rate: 4.5000e-05
Epoch 26/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 38s 897ms/step - accuracy: 0.9300 - loss: 0.2464 - val_accuracy: 0.9327 - val_loss: 0.2020 - learning_rate: 4.5000e-05
Epoch 27/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 916ms/step - accuracy: 0.9330 - loss: 0.2537 - val_accuracy: 0.9327 - val_loss: 0.2019 - learning_rate: 4.5000e-05
Epoch 28/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 39s 917ms/step - accuracy: 0.9286 - loss: 0.2546 - val_accuracy: 0.9310 - val_loss: 0.2070 - learning_rate: 4.5000e-05
Epoch 29/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 753ms/step - accuracy: 0.9378 - loss: 0.2138

43/43 ━━━━━━━━━━━━━━━━━━━━ 40s 923ms/step - accuracy: 0.9249 - loss: 0.2460 - val_accuracy: 0.9381 - val_loss: 0.1989 - learning_rate: 4.5000e-05
Epoch 30/30
43/43 ━━━━━━━━━━━━━━━━━━━━ 38s 899ms/step - accuracy: 0.9227 - loss: 0.2502 - val_accuracy: 0.9345 - val_loss: 0.2008 - learning_rate: 4.5000e-05


# **Finetune**

In [10]:
base_model.trainable = True

food_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=======Start Fine-tuning=======")

fine_result = food_model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS_FINE,
    callbacks=[save_best, stop_early, reduce_lr]
)

=======Start Fine-tuning=======
Epoch 1/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 353s 4s/step - accuracy: 0.8535 - loss: 0.4414 - val_accuracy: 0.8478 - val_loss: 0.4877 - learning_rate: 1.0000e-04
Epoch 2/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 828ms/step - accuracy: 0.9417 - loss: 0.1851

43/43 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.9308 - loss: 0.2069 - val_accuracy: 0.9575 - val_loss: 0.1379 - learning_rate: 1.0000e-04
Epoch 3/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.9627 - loss: 0.1337

43/43 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.9558 - loss: 0.1466 - val_accuracy: 0.9717 - val_loss: 0.0788 - learning_rate: 1.0000e-04
Epoch 4/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.9661 - loss: 0.1019 - val_accuracy: 0.9398 - val_loss: 0.1734 - learning_rate: 1.0000e-04
Epoch 5/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 867ms/step - accuracy: 0.9781 - loss: 0.0779

43/43 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.9794 - loss: 0.0793 - val_accuracy: 0.9894 - val_loss: 0.0423 - learning_rate: 1.0000e-04
Epoch 6/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 890ms/step - accuracy: 0.9751 - loss: 0.0720

43/43 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.9794 - loss: 0.0705 - val_accuracy: 0.9912 - val_loss: 0.0276 - learning_rate: 1.0000e-04
Epoch 7/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 862ms/step - accuracy: 0.9882 - loss: 0.0420

43/43 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.9860 - loss: 0.0485 - val_accuracy: 0.9929 - val_loss: 0.0286 - learning_rate: 1.0000e-04
Epoch 8/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 826ms/step - accuracy: 0.9937 - loss: 0.0334
Epoch 8: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
43/43 ━━━━━━━━━━━━━━━━━━━━ 41s 952ms/step - accuracy: 0.9897 - loss: 0.0393 - val_accuracy: 0.9912 - val_loss: 0.0309 - learning_rate: 1.0000e-04
Epoch 9/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 829ms/step - accuracy: 0.9910 - loss: 0.0336

43/43 ━━━━━━━━━━━━━━━━━━━━ 43s 988ms/step - accuracy: 0.9897 - loss: 0.0321 - val_accuracy: 0.9965 - val_loss: 0.0172 - learning_rate: 3.0000e-05
Epoch 10/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 42s 976ms/step - accuracy: 0.9934 - loss: 0.0312 - val_accuracy: 0.9965 - val_loss: 0.0137 - learning_rate: 3.0000e-05
Epoch 11/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 839ms/step - accuracy: 0.9896 - loss: 0.0269

43/43 ━━━━━━━━━━━━━━━━━━━━ 43s 1000ms/step - accuracy: 0.9897 - loss: 0.0274 - val_accuracy: 0.9982 - val_loss: 0.0139 - learning_rate: 3.0000e-05
Epoch 12/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 850ms/step - accuracy: 0.9947 - loss: 0.0260
Epoch 12: ReduceLROnPlateau reducing learning rate to 8.999999772640877e-06.
43/43 ━━━━━━━━━━━━━━━━━━━━ 42s 975ms/step - accuracy: 0.9948 - loss: 0.0277 - val_accuracy: 0.9965 - val_loss: 0.0152 - learning_rate: 3.0000e-05
Epoch 13/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 42s 989ms/step - accuracy: 0.9956 - loss: 0.0222 - val_accuracy: 0.9965 - val_loss: 0.0139 - learning_rate: 9.0000e-06
Epoch 14/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 43s 994ms/step - accuracy: 0.9919 - loss: 0.0250 - val_accuracy: 0.9982 - val_loss: 0.0109 - learning_rate: 9.0000e-06
Epoch 15/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 42s 988ms/step - accuracy: 0.9993 - loss: 0.0140 - val_accuracy: 0.9982 - val_loss: 0.0111 - learning_rate: 9.0000e-06
Epoch 16/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 868ms/step - accuracy: 0.99

In [11]:
print("=======Testing=======")

test_loss, test_acc = food_model.evaluate(test_loader)

print("Test accuracy :", test_acc)
print("Test loss :", test_loss)

=======Testing=======
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 335ms/step - accuracy: 0.9918 - loss: 0.0237
Test accuracy : 0.9918367266654968
Test loss : 0.0237442534416914


In [12]:
import json
from google.colab import files

# =====SAVE METRICS=====
save_training_result = {
    'train_acc': train_result.history['accuracy'],
    'val_acc': train_result.history['val_accuracy'],
    'train_loss': train_result.history['loss'],
    'val_loss': train_result.history['val_loss']
}

metrics_path = '/content/save_training_result.json'

with open(metrics_path, 'w') as f:
    json.dump(save_training_result, f, indent=2)

print('save_training_result.json complete')

# =====SAVE CLASS MAP=====
class_map_path = '/content/class_map.json'
with open(class_map_path, 'w') as f:
    json.dump(train_loader.class_indices, f, indent=2)

print('class_map.json complete')

files.download(metrics_path)
files.download(MODEL_PATH)
files.download(class_map_path)

save_training_result.json complete
class_map.json complete


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
food_model.save('food_classifier.keras')

from google.colab import files
files.download('food_classifier.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>